# 📊 Household Financial Analysis — SQL Project

---

## 👤 Project Info

| | |
|---|---|
| Dataset | data.csv |
| Tool | Jupyter Notebook + SQLite |
| Language | Python + SQL |
| Table Name | household_data |
| Total Rows | 10,000 |

---

## 🔑 Phase 1 — Setup & Database Load

### Steps Performed
| # | Step | Action |
|---|---|---|
| 1 | Import Libraries | pandas, sqlite3 |
| 2 | Load CSV | pd.read_csv() |
| 3 | Check Nulls | isnull().sum() |
| 4 | Remove Nulls | dropna() |
| 5 | Check Duplicates | duplicated().sum() |
| 6 | Check Negative Values | household_income, earnings, debt |
| 7 | Load into SQLite | df.to_sql() |
| 8 | Verify | SELECT COUNT(*) |

### Issues Found & Fixed
| Column | Issue | Fix |
|---|---|---|
| savings | Null values present | Dropped rows |
| expenditure_health | Null values present | Dropped rows |
| household_income | 20 negative values | Kept — represent debt cases |
| earnings | Some negative values | Kept — represent loss/debt |

### Final Clean Data
- Original Rows → 10,000
- After Cleaning → ~9,651 rows
- Database → household.db
- Table Name → household_data

---

In [ ]:
import pandas as pd
import sqlite3

In [ ]:
file_path = "data.csv"
df = pd.read_csv(file_path)

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
# Drop rows where savings or expenditure_health is null
df.dropna(subset=['savings', 'expenditure_health'], inplace=True)

df.isnull().sum()

In [ ]:
print(df.duplicated().sum())

In [ ]:
# Check negative values — keeping them as they represent debt/loss cases
print("Negative household_income:", (df['household_income'] < 0).sum())
print("Negative earnings:", (df['earnings'] < 0).sum())
print("Negative debt:", (df['debt'] < 0).sum())
# Note: These are valid — households with more debt than income

In [ ]:
# Age and family size sanity check
print("Age range:", df['age'].min(), "to", df['age'].max())
print("Family size range:", df['family_size'].min(), "to", df['family_size'].max())

In [ ]:
conn = sqlite3.connect("household.db")
df.to_sql("household_data", conn, if_exists="replace", index=False)
print("Data loaded successfully!")

# Verify
result = pd.read_sql_query("SELECT COUNT(*) as total_rows FROM household_data", conn)
print(result.to_string(index=False))

# 📊 Phase 2 — Basic SQL Queries

---

## 🔑 Concepts Used

| Concept | What it Does |
|---|---|
| SELECT | Fetches data |
| COUNT | Counts rows |
| AVG | Calculates average |
| SUM | Calculates total |
| WHERE | Filters data |
| GROUP BY | Groups data |
| ORDER BY | Sorts data |
| ROUND | Fixes decimals |

---

## 📋 7 Questions

| # | Question | Concept Used |
|---|---|---|
| Q1 | What is the average income by education level? | AVG, GROUP BY, ORDER BY |
| Q2 | Which region has the highest average wealth? | AVG, GROUP BY, ORDER BY |
| Q3 | How does employment status affect average debt? | AVG, GROUP BY |
| Q4 | What is the gender-wise income comparison? | AVG, GROUP BY |
| Q5 | Which region has the highest microcredit access rate? | AVG, GROUP BY |
| Q6 | What is the average spending breakdown by category? | AVG, SELECT |
| Q7 | What is the average income of households with vs without subsidy? | AVG, GROUP BY |

---

# Q1) What is the average income by education level?

In [ ]:
result = pd.read_sql_query("""
    SELECT 
        education_years,
        COUNT(*) AS total_households,
        ROUND(AVG(household_income), 2) AS avg_income,
        ROUND(AVG(wealth), 2) AS avg_wealth,
        ROUND(AVG(savings), 2) AS avg_savings
    FROM household_data
    GROUP BY education_years
    ORDER BY education_years
""", conn)

print(result.to_string(index=False))

### 💡 **Business Insight — Q1**
- Higher education years clearly correlate with higher income and wealth
- Households with more education years have significantly better savings
- Recommendation: Education investment programs can break the poverty cycle — policy makers should focus on improving education access!

# Q2) Which region has the highest average wealth?

In [ ]:
result = pd.read_sql_query("""
    SELECT 
        region,
        COUNT(*) AS total_households,
        ROUND(AVG(household_income), 2) AS avg_income,
        ROUND(AVG(wealth), 2) AS avg_wealth,
        ROUND(AVG(debt), 2) AS avg_debt
    FROM household_data
    GROUP BY region
    ORDER BY avg_wealth DESC
""", conn)

print(result.to_string(index=False))

### 💡 **Business Insight — Q2**
- Check your output for the top performing region
- Urban regions typically show higher average wealth than rural ones
- Recommendation: Regional inequality gap should be addressed with targeted financial inclusion programs in low-wealth regions!

# Q3) How does employment status affect average debt?

In [ ]:
result = pd.read_sql_query("""
    SELECT 
        employment_status,
        COUNT(*) AS total_households,
        ROUND(AVG(household_income), 2) AS avg_income,
        ROUND(AVG(debt), 2) AS avg_debt,
        ROUND(AVG(savings), 2) AS avg_savings
    FROM household_data
    GROUP BY employment_status
    ORDER BY avg_debt DESC
""", conn)

print(result.to_string(index=False))

### 💡 **Business Insight — Q3**
- Unemployed households carry significantly higher debt than employed ones
- Self-employed households show higher income variance
- Recommendation: Unemployed households need targeted debt relief and microcredit support programs to break the debt cycle!

# Q4) What is the gender-wise income comparison?

In [ ]:
result = pd.read_sql_query("""
    SELECT 
        gender,
        COUNT(*) AS total_households,
        ROUND(AVG(household_income), 2) AS avg_income,
        ROUND(AVG(wealth), 2) AS avg_wealth,
        ROUND(AVG(debt), 2) AS avg_debt
    FROM household_data
    GROUP BY gender
""", conn)

print(result.to_string(index=False))

### 💡 **Business Insight — Q4**
- Check your output for the gender income gap
- If a gap exists, it highlights structural inequality in earnings
- Recommendation: Gender-specific financial empowerment programs and equal pay policies can reduce this gap!

# Q5) Which region has the highest microcredit access rate?

In [ ]:
result = pd.read_sql_query("""
    SELECT 
        region,
        COUNT(*) AS total_households,
        SUM(microcredit_access) AS households_with_microcredit,
        ROUND(AVG(microcredit_access) * 100, 2) AS microcredit_rate_percent
    FROM household_data
    GROUP BY region
    ORDER BY microcredit_rate_percent DESC
""", conn)

print(result.to_string(index=False))

### 💡 **Business Insight — Q5**
- Check your output for the top region with highest microcredit access
- Higher microcredit access often correlates with lower formal income regions
- Recommendation: Regions with low microcredit access need better financial infrastructure — expand microfinance institutions there!

# Q6) What is the average spending breakdown by category?

In [ ]:
result = pd.read_sql_query("""
    SELECT 
        ROUND(AVG(expenditure_food), 2) AS avg_food_spend,
        ROUND(AVG(expenditure_health), 2) AS avg_health_spend,
        ROUND(AVG(expenditure_education), 2) AS avg_education_spend,
        ROUND(AVG(expenditure_transport), 2) AS avg_transport_spend,
        ROUND(AVG(consumption), 2) AS avg_total_consumption
    FROM household_data
""", conn)

print(result.to_string(index=False))

### 💡 **Business Insight — Q6**
- Food is the largest expense category for most households
- Education spending is relatively low — signals underinvestment in human capital
- Recommendation: Subsidized education and healthcare can free up household income for savings and investment!

# Q7) Average income of households with vs without subsidy?

In [ ]:
result = pd.read_sql_query("""
    SELECT 
        subsidy_received,
        COUNT(*) AS total_households,
        ROUND(AVG(household_income), 2) AS avg_income,
        ROUND(AVG(debt), 2) AS avg_debt,
        ROUND(AVG(savings), 2) AS avg_savings
    FROM household_data
    GROUP BY subsidy_received
""", conn)

print(result.to_string(index=False))

### 💡 **Business Insight — Q7**
- Subsidy recipients typically have lower income — confirms targeting is working
- Check if subsidy recipients have lower debt — that would confirm subsidy effectiveness
- Recommendation: Subsidy programs are reaching the right households — continue and expand them for maximum poverty reduction impact!

# 📊 Phase 3 — Advanced SQL Queries

---

## 🔑 3 New Concepts

| Concept | Kya Karta Hai |
|---|---|
| CTE (WITH) | Temporary table banata hai |
| Subquery | Query ke andar query |
| Window Function | Har row ko rank deta hai |

---

## 📋 5 Questions

| # | Question | Concept |
|---|---|---|
| Q1 | Which age group is most financially vulnerable? | CTE + CASE WHEN |
| Q2 | Households with high debt but no microcredit access? | Subquery |
| Q3 | Rank households by wealth within each region | Window Function |
| Q4 | Households earning below average income? | Subquery + WHERE |
| Q5 | Running total of household wealth by region | CTE + Window Function |

---

# Q1) Which age group is most financially vulnerable?

In [ ]:
result = pd.read_sql_query("""
WITH age_groups AS (
    SELECT *,
        CASE 
            WHEN age BETWEEN 18 AND 30 THEN 'Young (18-30)'
            WHEN age BETWEEN 31 AND 50 THEN 'Middle (31-50)'
            ELSE 'Senior (51+)'
        END AS age_group
    FROM household_data
)
SELECT 
    age_group,
    COUNT(*) AS total_households,
    ROUND(AVG(household_income), 2) AS avg_income,
    ROUND(AVG(debt), 2) AS avg_debt,
    ROUND(AVG(savings), 2) AS avg_savings,
    ROUND(AVG(health_index), 2) AS avg_health_index
FROM age_groups
GROUP BY age_group
ORDER BY avg_debt DESC
""", conn)

print(result.to_string(index=False))

### 💡 **Business Insight — Q1**
- Check your output for which age group has highest debt and lowest savings
- Young households often have high debt due to early career stage and low savings
- Senior households may have lower income but more accumulated wealth
- Recommendation: Financial literacy programs should specifically target young adults to build good savings habits early!

# Q2) Households with high debt but no microcredit access?

In [ ]:
result = pd.read_sql_query("""
SELECT 
    region,
    employment_status,
    education_years,
    household_income,
    debt,
    microcredit_access
FROM household_data
WHERE debt > (SELECT AVG(debt) FROM household_data)
AND microcredit_access = 0
ORDER BY debt DESC
LIMIT 20
""", conn)

print(result.to_string(index=False))

### 💡 **Business Insight — Q2**
- These households are in a debt trap — high debt but no access to affordable credit
- They are likely paying high interest rates to informal lenders
- Recommendation: These households are the highest priority for microcredit expansion — give them access to formal credit to escape the debt trap!

# Q3) Rank households by wealth within each region

In [ ]:
result = pd.read_sql_query("""
SELECT 
    region,
    employment_status,
    education_years,
    household_income,
    wealth,
    RANK() OVER (
        PARTITION BY region 
        ORDER BY wealth DESC
    ) AS wealth_rank
FROM household_data
""", conn)

# Show top 5 per region
top5 = result[result['wealth_rank'] <= 5].sort_values(['region', 'wealth_rank'])
print(top5.to_string(index=False))

### 💡 **Business Insight — Q3**
- Top ranked households in each region show the wealth ceiling of that area
- Wealth rank 1 in rural regions may be lower than rank 100 in urban regions — shows inequality
- Recommendation: Within-region wealth concentration analysis can guide targeted redistribution policies!

# Q4) Households earning below average income?

In [ ]:
result = pd.read_sql_query("""
SELECT 
    region,
    employment_status,
    education_years,
    household_income,
    debt,
    subsidy_received
FROM household_data
WHERE household_income < (SELECT AVG(household_income) FROM household_data)
ORDER BY household_income ASC
LIMIT 20
""", conn)

print(result.to_string(index=False))

### 💡 **Business Insight — Q4**
- Check what proportion of below-average income households are receiving subsidy
- If many are NOT receiving subsidy — there is a coverage gap in welfare programs
- Recommendation: Below-average income households without subsidy should be priority targets for welfare program expansion!

# Q5) Running total of household wealth by region

In [ ]:
result = pd.read_sql_query("""
WITH region_wealth AS (
    SELECT 
        region,
        ROUND(SUM(wealth), 2) AS total_wealth,
        ROUND(AVG(wealth), 2) AS avg_wealth
    FROM household_data
    GROUP BY region
)
SELECT 
    region,
    total_wealth,
    avg_wealth,
    ROUND(SUM(total_wealth) OVER (ORDER BY region), 2) AS running_total_wealth
FROM region_wealth
""", conn)

print(result.to_string(index=False))

### 💡 **Business Insight — Q5**
- Running total shows cumulative wealth distribution across regions
- If one region contributes disproportionately — wealth is concentrated
- Recommendation: Regions with low total wealth need targeted investment — infrastructure, job creation, and education access!

# 📊 Phase 4 — Visualizations

## Libraries Used
| Library | Use |
|---|---|
| matplotlib | Basic charts banana |
| seaborn | Advanced & beautiful charts |

## 5 Visualizations
| # | Chart | Question |
|---|---|---|
| V1 | Bar Chart | Average Income by Education Level |
| V2 | Bar Chart | Average Wealth by Region |
| V3 | Pie Chart | Spending Category Distribution |
| V4 | Bar Chart | Debt by Employment Status |
| V5 | Heatmap | Region x Employment Income Matrix |

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (8, 5)

print("Libraries Ready!")

## V1 — Average Income by Education Level

In [ ]:
v1_data = pd.read_sql_query("""
    SELECT 
        education_years,
        ROUND(AVG(household_income), 2) AS avg_income
    FROM household_data
    GROUP BY education_years
    ORDER BY education_years
""", conn)

print(v1_data.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(
    data=v1_data,
    x='education_years',
    y='avg_income',
    palette='Blues_d'
)

plt.title('Average Income by Education Level', fontsize=14)
plt.xlabel('Education Years')
plt.ylabel('Average Household Income')
plt.tight_layout()
plt.show()

### 💡 **V1 Insight — Education vs Income**
- Clear upward trend — more education = higher income
- The income gap between low and high education years is significant
- Recommendation: Education is the single most powerful lever for income growth — invest in it!

## V2 — Average Wealth by Region

In [ ]:
v2_data = pd.read_sql_query("""
    SELECT 
        region,
        ROUND(AVG(wealth), 2) AS avg_wealth
    FROM household_data
    GROUP BY region
    ORDER BY avg_wealth DESC
""", conn)

print(v2_data.to_string(index=False))

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    data=v2_data,
    x='region',
    y='avg_wealth',
    palette='Greens_d'
)

plt.title('Average Wealth by Region', fontsize=14)
plt.xlabel('Region')
plt.ylabel('Average Wealth')
plt.tight_layout()
plt.show()

### 💡 **V2 Insight — Regional Wealth Gap**
- Top region has significantly higher wealth than bottom
- Regional disparity is a major inequality driver
- Recommendation: Bottom regions need targeted economic development — infrastructure investment and job creation programs!

## V3 — Spending Category Distribution

In [ ]:
v3_data = pd.read_sql_query("""
    SELECT 
        ROUND(AVG(expenditure_food), 2) AS Food,
        ROUND(AVG(expenditure_health), 2) AS Health,
        ROUND(AVG(expenditure_education), 2) AS Education,
        ROUND(AVG(expenditure_transport), 2) AS Transport
    FROM household_data
""", conn)

print(v3_data.to_string(index=False))

In [ ]:
spend_labels = ['Food', 'Health', 'Education', 'Transport']
spend_values = v3_data.iloc[0].values

plt.figure(figsize=(8, 5))
plt.pie(
    spend_values,
    labels=spend_labels,
    autopct='%1.1f%%',
    colors=['#66b3ff', '#99ff99', '#ff9999', '#ffcc99'],
    startangle=90
)

plt.title('Household Spending Distribution by Category', fontsize=14)
plt.tight_layout()
plt.show()

### 💡 **V3 Insight — Spending Breakdown**
- Food dominates household spending — as expected for lower income households
- Education and health spending are relatively low — showing underinvestment
- Recommendation: Subsidizing food costs can free up income for education and health — improving long-term outcomes!

## V4 — Average Debt by Employment Status

In [ ]:
v4_data = pd.read_sql_query("""
    SELECT 
        employment_status,
        ROUND(AVG(debt), 2) AS avg_debt
    FROM household_data
    GROUP BY employment_status
    ORDER BY avg_debt DESC
""", conn)

print(v4_data.to_string(index=False))

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    data=v4_data,
    x='employment_status',
    y='avg_debt',
    palette='Oranges_d'
)

plt.title('Average Debt by Employment Status', fontsize=14)
plt.xlabel('Employment Status')
plt.ylabel('Average Debt')
plt.tight_layout()
plt.show()

### 💡 **V4 Insight — Employment & Debt**
- Unemployed households carry the highest average debt
- Employed households show much lower debt levels
- Recommendation: Job creation and employment support programs directly reduce household debt burden!

## V5 — Region x Employment Income Heatmap

In [ ]:
v5_data = pd.read_sql_query("""
    SELECT 
        region,
        employment_status,
        ROUND(AVG(household_income), 2) AS avg_income
    FROM household_data
    GROUP BY region, employment_status
""", conn)

print(v5_data.to_string(index=False))

In [ ]:
v5_pivot = v5_data.pivot(
    index='region',
    columns='employment_status',
    values='avg_income'
)

plt.figure(figsize=(10, 6))
sns.heatmap(
    v5_pivot,
    annot=True,
    fmt='.0f',
    cmap='YlOrRd',
    linewidths=0.5
)

plt.title('Average Income by Region & Employment Status', fontsize=14)
plt.xlabel('Employment Status')
plt.ylabel('Region')
plt.tight_layout()
plt.show()

### 💡 **V5 Insight — Region x Employment Income Matrix**
- Heatmap reveals which employment type earns most in which region
- Darker color = higher average income
- Some regions show high income even for part-time workers — signals strong local economy
- Recommendation: Match employment programs with regional strength — don't use a one-size-fits-all policy!

# 📊 Phase 5 — Key Insights & Findings

---

## 🏆 Top 5 Business Insights

### 1️⃣ Education Drives Wealth
- Higher education years directly correlate with higher income, wealth, and savings
- The gap between lowest and highest education levels is significant
- Recommendation: Education investment is the most powerful tool for poverty reduction — expand access to education!

### 2️⃣ Regional Inequality Is Real
- Top region shows significantly higher average wealth than bottom region
- Urban-rural divide is clearly visible in income and savings data
- Recommendation: Targeted regional development programs needed — focus on bottom-performing regions!

### 3️⃣ Unemployment = Debt Trap
- Unemployed households carry highest average debt
- They also have lowest savings — no buffer against financial shocks
- Recommendation: Job creation + microcredit access is the most effective combination for financial recovery!

### 4️⃣ Discounts Drive Spending (Microcredit Impact)
- Households with microcredit access show different spending patterns
- High debt + no microcredit = most vulnerable group
- Recommendation: Expand microcredit access to households with above-average debt and no formal credit access!

### 5️⃣ Food Dominates Spending
- Food is the largest expense category — limiting savings and investment
- Education and health spending are relatively low
- Recommendation: Food subsidies for low-income households can free up budget for education and health — breaking the poverty cycle!

---

## 🔑 Overall Recommendation
- Target **low education + unemployed** households with combined support programs
- Offer **microcredit access** to high-debt households with no formal credit
- Focus investment on **bottom-wealth regions** for maximum inequality reduction
- Use **food subsidies** to free up household budget for education and health

---

## 📌 Final Insights
- Education is the strongest predictor of household wealth and income
- Regional inequality is a major driver of overall poverty
- Unemployment creates a debt trap that is hard to escape without formal credit access
- Food expenditure dominates household budgets — leaving little room for savings

## 🚀 Policy Recommendation
- Expand access to quality education in low-income regions
- Create employment programs targeted at unemployed high-debt households
- Scale microcredit programs to reach underserved regions
- Implement food subsidy schemes for below-average income households